In [ ]:
# !pip install -q "numpy>=2.0,<2.3"
# !pip install -q "opencv-python-headless>=4.9.0"
# !pip install -q pytorch-lightning==1.9.5
# !pip install -q diffusers==0.21.4 transformers==4.35.0 accelerate==0.24.1 safetensors==0.4.0
# !pip install -q pillow einops omegaconf open_clip_torch kornia clean-fid
# !pip install -q "gradio<4.0.0"

# !pip uninstall -y numpy pandas -q
# !pip install -q numpy==1.26.4 pandas==2.1.4

In [ ]:
import os

print("=== INPUT DATASETS ===")
for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

In [ ]:
import shutil
import os

SRC = "/kaggle/input/datasets/vuong2901/stable-viton-source"
DST = "/kaggle/working/project"

if os.path.exists(DST):
    shutil.rmtree(DST)

shutil.copytree(SRC, DST)
print("✅ Source copied")

In [ ]:
CKPT_SRC = "/kaggle/input/datasets/vuong2901/stable-viton-ckpt/ckpts/VITONHD.ckpt"
CKPT_DST = "/kaggle/working/project/checkpoints/stablevton/ckpts/VITONHD.ckpt"

os.makedirs(os.path.dirname(CKPT_DST), exist_ok=True)

if not os.path.exists(CKPT_DST):
    os.symlink(CKPT_SRC, CKPT_DST)

print("✅ Checkpoint linked")

In [ ]:
SRC_ASSETS = "/kaggle/input/datasets/vuong2901/stable-viton-demo/offline_assets/offline_assets"
DST_ASSETS = "/kaggle/working/offline_assets"

if os.path.exists(DST_ASSETS):
    shutil.rmtree(DST_ASSETS)

shutil.copytree(SRC_ASSETS, DST_ASSETS)
print("✅ offline_assets ready")

In [ ]:
APP_SRC = "/kaggle/input/datasets/vuong2901/stable-viton-demo/app_online.py"
APP_DST = "/kaggle/working/app_online.py"

shutil.copy(APP_SRC, APP_DST)

print("✅ app_online.py copied")

In [ ]:
import sys
from pathlib import Path

sys.path.append("/kaggle/working")

import app_online

print("✅ app_online imported")

app_online.BASE_DIR = Path("/kaggle/working")

app_online.RUNTIME_DIR = app_online.BASE_DIR / "runtime"
app_online.EXTRACT_DIR = app_online.RUNTIME_DIR / "offline_assets"
app_online.DATA_ROOT = app_online.RUNTIME_DIR / "data" / "viton_hd"
app_online.OUTPUT_ROOT = app_online.RUNTIME_DIR / "outputs"

app_online.STABLEVITON_ROOT = Path("/kaggle/working/project/StableVITON")
app_online.CKPT_PATH = Path("/kaggle/working/project/checkpoints/stablevton/ckpts/VITONHD.ckpt")
app_online.CONFIG_PATH = app_online.STABLEVITON_ROOT / "configs" / "VITONHD.yaml"

print("✅ app_online patched")

In [ ]:
import json
import shutil
import subprocess
import sys
import numpy as np
import cv2
from pathlib import Path
from PIL import Image

# =========================
# LOAD META
# =========================

ASSET_ROOT = Path("/kaggle/working/offline_assets")

# Tìm file meta.json
meta_paths = list(ASSET_ROOT.rglob("meta.json"))
if not meta_paths:
    raise FileNotFoundError("Không tìm thấy file meta.json trong thư mục offline_assets.")
meta_path = meta_paths[0]

with open(meta_path, "r") as f:
    meta = json.load(f)

artifact_root = meta_path.parents[2]


# =========================
# LOAD DATA
# =========================
person_img = Image.open(artifact_root / meta["person_image"]).convert("RGB")
cloth_img = Image.open(artifact_root / meta["cloth_image"]).convert("RGB")
agnostic_img = Image.open(artifact_root / meta["agnostic"]).convert("RGB")
agnostic_mask = Image.open(artifact_root / meta["agnostic_mask"]).convert("L")
densepose_img = Image.open(artifact_root / meta["densepose"]).convert("RGB")
cloth_mask = Image.open(artifact_root / meta["cloth_mask"]).convert("L")

# =========================
# PATCH FIX LỖI APP_ONLINE (V10 - Tinh chỉnh Mask ghép túi)
# =========================

def patched_prepare_viton_test_sample(
    person_img, cloth_img, cloth_mask, agnostic_img, agnostic_mask, densepose_img, sample_name="00000_00.jpg"
):
    test_root = app_online.DATA_ROOT / "test"
    name = sample_name 

    folders = ["image", "cloth", "cloth-mask", "agnostic-mask", "agnostic-v3.2", "image-densepose"]

    if app_online.DATA_ROOT.exists():
        shutil.rmtree(app_online.DATA_ROOT)

    for folder in folders:
        (test_root / folder).mkdir(parents=True, exist_ok=True)

    # 1. TẠO MASK MỞ RỘNG (GỘP TÚI)
    agnostic_mask_old_np = np.array(agnostic_mask.convert("L")) > 127
    
    bag_mask_sam_path = artifact_root / meta["object_masks"]["bag"]
    if bag_mask_sam_path.exists():
        object_mask_sam_np = np.array(Image.open(bag_mask_sam_path).convert("L")) > 127
        combined_mask_np = np.maximum(agnostic_mask_old_np, object_mask_sam_np)
    else:
        combined_mask_np = agnostic_mask_old_np
        
    # Mở rộng mask
    kernel = np.ones((25, 25), np.uint8) 
    expanded_mask_np = cv2.dilate(combined_mask_np.astype(np.uint8), kernel, iterations=1)
    
    agnostic_mask_safe = Image.fromarray((expanded_mask_np * 255).astype(np.uint8))
    
    # 2. TẠO AGNOSTIC IMAGE NỀN XÁM (XÓA SẠCH TÚI)
    person_np = np.array(person_img).copy()
    
    mask_boolean = (expanded_mask_np > 0)
    person_np[mask_boolean] = 127 
    
    agnostic_img_safe = Image.fromarray(person_np)

    # 3. LƯU FILE
    person_img.save(test_root / "image" / name)
    agnostic_img_safe.save(test_root / "agnostic-v3.2" / name)
    densepose_img.save(test_root / "image-densepose" / name)
    cloth_img.save(test_root / "cloth" / name)
    cloth_mask.save(test_root / "cloth-mask" / name) 
    
    agnostic_mask_safe.save(test_root / "agnostic-mask" / name.replace(".jpg", "_mask.png"))
    agnostic_mask_safe.save(test_root / "agnostic-mask" / name.replace(".jpg", ".png"))

    with open(app_online.DATA_ROOT / "test_pairs.txt", "w") as f:
        f.write(f"{name} {name}\n")

def patched_run_stableviton():
    if app_online.OUTPUT_ROOT.exists():
        shutil.rmtree(app_online.OUTPUT_ROOT)
    app_online.OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        "inference.py",
        "--config_path", str(app_online.CONFIG_PATH),
        "--batch_size", "1",
        "--model_load_path", str(app_online.CKPT_PATH),
        "--save_dir", str(app_online.OUTPUT_ROOT),
        "--data_root_dir", str(app_online.DATA_ROOT)
    ]
    print("Running StableVITON (Patched V10)...")
    result = subprocess.run(cmd, cwd=str(app_online.STABLEVITON_ROOT), capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError("StableVITON failed")


# HÀM MỚI: TINH CHỈNH MASK VÀ GHÉP TÚI (Alpha Blending Nâng Cao)
def refined_restore_occluder(original_img, result_img, object_mask, erode_iter=2, blur_ksize=5):
    """
    Thu hẹp mask để cắt bỏ viền trắng/đen thừa, sau đó làm mờ nhẹ.
    Ép kích thước ảnh gốc và mask về bằng với ảnh kết quả trước khi xử lý.
    """
    target_size = result_img.size # Lấy size (width, height) của ảnh output
    
    # ÉP SIZE CHO KHỚP NHAU
    orig_resized = original_img.resize(target_size, Image.Resampling.LANCZOS)
    mask_resized = object_mask.resize(target_size, Image.Resampling.NEAREST)
    
    orig_np = np.array(orig_resized).astype(np.float32)
    res_np = np.array(result_img).astype(np.float32)
    mask_np = np.array(mask_resized.convert("L"))
    
    # 1. Thu hẹp mask (Erode) để gọt bớt phần viền thừa
    kernel_erode = np.ones((3, 3), np.uint8)
    eroded_mask = cv2.erode(mask_np, kernel_erode, iterations=erode_iter)
    
    # 2. Làm mờ viền (Blur)
    if blur_ksize > 0:
        if blur_ksize % 2 == 0:
            blur_ksize += 1
        blurred_mask = cv2.GaussianBlur(eroded_mask, (blur_ksize, blur_ksize), 0)
    else:
        blurred_mask = eroded_mask

    # 3. Alpha Blending
    alpha = blurred_mask.astype(np.float32) / 255.0
    alpha = alpha[..., np.newaxis] 
    
    blended_np = orig_np * alpha + res_np * (1.0 - alpha)
    
    Image.fromarray(blurred_mask).save("/kaggle/working/refined_mask_debug.png")
    
    return Image.fromarray(blended_np.astype(np.uint8))

# GHI ĐÈ HÀM
app_online.prepare_viton_test_sample = patched_prepare_viton_test_sample
app_online.run_stableviton = patched_run_stableviton
print("✅ Đã cấu hình các hàm vá lỗi V10!")


# =========================
# PREPARE DATA
# =========================
app_online.prepare_viton_test_sample(
    person_img,
    cloth_img,
    cloth_mask,
    agnostic_img,
    agnostic_mask,
    densepose_img,
    sample_name="00000_00.jpg"
)


# =========================
# RUN MODEL
# =========================
app_online.run_stableviton()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def show_debug_bundle(person, agnostic, agnostic_mask, densepose, cloth, cloth_mask):
    fig, axs = plt.subplots(2, 3, figsize=(12, 10))
    axs[0,0].imshow(person); axs[0,0].set_title("person")
    axs[0,1].imshow(agnostic); axs[0,1].set_title("agnostic")
    axs[0,2].imshow(agnostic_mask, cmap="gray"); axs[0,2].set_title("agnostic_mask")
    axs[1,0].imshow(densepose); axs[1,0].set_title("densepose")
    axs[1,1].imshow(cloth); axs[1,1].set_title("cloth")
    axs[1,2].imshow(cloth_mask, cmap="gray"); axs[1,2].set_title("cloth_mask")
    for ax in axs.flatten():
        ax.axis("off")
    plt.tight_layout()
    plt.show()

def overlay_mask(img, mask, title="overlay"):
    img_np = np.array(img).astype(np.float32)
    mask_np = (np.array(mask) > 127).astype(np.float32)
    overlay = img_np.copy()
    overlay[..., 0] = overlay[..., 0] * (1 - mask_np) + 255 * mask_np
    plt.imshow(overlay.astype(np.uint8)); plt.title(title); plt.axis("off"); plt.show()

def print_stats(name, img):
    arr = np.array(img)
    print(f"{name}: shape={arr.shape}, min={arr.min()}, max={arr.max()}, mean={arr.mean():.2f}")


# ===== RUN DEBUG =====
print("=== DEBUG INPUTS ===")
print_stats("person", person_img)
print_stats("agnostic", agnostic_img)
print_stats("agnostic_mask", agnostic_mask)
print_stats("densepose", densepose_img)
print_stats("cloth", cloth_img)
print_stats("cloth_mask", cloth_mask)

show_debug_bundle(person_img, agnostic_img, agnostic_mask, densepose_img, cloth_img, cloth_mask)
overlay_mask(person_img, agnostic_mask, "agnostic_mask overlay")
overlay_mask(cloth_img, cloth_mask, "cloth_mask overlay")

In [ ]:
import matplotlib.pyplot as plt

# =========================
# LOAD RESULT & RESTORE BAG
# =========================
result_path = app_online.find_latest_result()
result_img = Image.open(result_path).convert("RGB")

plt.imshow(result_img)
plt.title("RAW RESULT (before bag restore)")
plt.axis("off")
plt.show()

# =========================
# RESTORE BAG
# =========================
bag_mask = meta.get("object_masks", {}).get("bag", None)

if bag_mask:
    bag_path = artifact_root / bag_mask
    if bag_path.exists():
        object_mask_pil = Image.open(bag_path).convert("L")
        
        # SỬ DỤNG HÀM V10 (Đã loại bỏ hàm cũ để tránh xung đột)
        # Tùy chỉnh: erode_iter (tăng để gọt viền nhiều hơn), blur_ksize (tăng để mờ viền hơn)
        result_img = refined_restore_occluder(
            person_img, 
            result_img, 
            object_mask_pil, 
            erode_iter=1,   # Tăng nhẹ độ gọt viền
            blur_ksize=5    # Giảm nhẹ độ mờ để tránh lan vào chữ
        )

In [ ]:
# =========================
# SAVE + SHOW
# =========================
final_path = "/kaggle/working/final.png"
result_img.save(final_path)

print("✅ DONE:", final_path)

plt.imshow(result_img)
plt.axis("off")
plt.title("Final Try-On Result (V10 Refined)")
plt.show()